In [6]:
import numpy as np
import pandas as pd

N_CUSTOMERS = 5000
rng = np.random.default_rng(123)

def lognorm_dollars(mean_log, sd_log):
    # Positive-skew dollar amounts (realistic-ish balances).
    return float(np.round(np.exp(rng.normal(mean_log, sd_log)), 2))

rows = []

for cid in range(1, N_CUSTOMERS + 1):
    # Simple synthetic "wealth" proxy (0..1) to make product mix realistic
    wealth_proxy = rng.random()

    # ---------- DEPOSITS ----------
    # Always include checking
    deposit_types = {"Checking"}

    # Add other deposit products with probabilities tied to wealth_proxy
    if rng.random() < (0.70 + 0.10 * wealth_proxy):
        deposit_types.add("Savings")
    if rng.random() < (0.35 + 0.25 * wealth_proxy):
        deposit_types.add("Money Market")
    if rng.random() < (0.22 + 0.33 * wealth_proxy):
        deposit_types.add("CD Short Term")
    if rng.random() < (0.12 + 0.22 * wealth_proxy):
        deposit_types.add("CD Long Term")

    # Deposit balances
    for pt in deposit_types:
        if pt == "Checking":
            bal = lognorm_dollars(8.6, 0.9)     # ~5k–20k typical
            dep_cat = "NIDDA"
        elif pt == "Savings":
            bal = lognorm_dollars(9.0, 1.0)
            dep_cat = "IBB"
        elif pt == "Money Market":
            bal = lognorm_dollars(9.4, 1.0)
            dep_cat = "IBB"
        elif pt == "CD Short Term":
            bal = lognorm_dollars(9.6, 1.0)
            dep_cat = "IBB"
        else:  # CD Long Term
            bal = lognorm_dollars(9.9, 1.05)
            dep_cat = "IBB"

        rows.append({
            "cust_id": cid,
            "product_group": "Deposit",
            "deposit_category": dep_cat,
            "product_type": pt,
            "balance": bal,
            "active_flag": "Y"
        })

    # ---------- CREDIT ----------
    # Credit card likelihood: moderate baseline + slightly higher for wealthier clients
    p_cc = np.clip(0.45 + 0.20 * wealth_proxy, 0.20, 0.85)
    has_cc = rng.random() < p_cc

    cc_bal = lognorm_dollars(7.8, 0.9) if has_cc else 0.0  # ~1k–6k typical

    rows.append({
        "cust_id": cid,
        "product_group": "Credit",
        "deposit_category": "N/A",
        "product_type": "Credit Card",
        "balance": cc_bal,
        "active_flag": "Y" if has_cc else "N"
    })

    # ---------- LOANS ----------
    # Loan likelihoods (home loan increases with wealth; auto/personal slightly more common mid/low)
    p_home = np.clip(0.12 + 0.35 * wealth_proxy, 0.05, 0.55)
    p_auto = np.clip(0.18 + 0.18 * (1 - wealth_proxy), 0.08, 0.45)
    p_pl   = np.clip(0.10 + 0.15 * (1 - wealth_proxy), 0.04, 0.30)

    has_home = rng.random() < p_home
    has_auto = rng.random() < p_auto
    has_pl   = rng.random() < p_pl

    rows.append({
        "cust_id": cid,
        "product_group": "Loan",
        "deposit_category": "N/A",
        "product_type": "Home Loan",
        "balance": lognorm_dollars(13.0, 0.45) if has_home else 0.0,  # ~300k–700k+
        "active_flag": "Y" if has_home else "N"
    })

    rows.append({
        "cust_id": cid,
        "product_group": "Loan",
        "deposit_category": "N/A",
        "product_type": "Auto Loan",
        "balance": lognorm_dollars(10.5, 0.55) if has_auto else 0.0,  # ~20k–60k
        "active_flag": "Y" if has_auto else "N"
    })

    rows.append({
        "cust_id": cid,
        "product_group": "Loan",
        "deposit_category": "N/A",
        "product_type": "Personal Loan",
        "balance": lognorm_dollars(9.8, 0.65) if has_pl else 0.0,      # ~8k–30k
        "active_flag": "Y" if has_pl else "N"
    })

# Build tables
fact_cust_product_v2 = pd.DataFrame(rows)

# Active-only
fact_cust_product_active_v2 = fact_cust_product_v2.query("active_flag == 'Y'").copy()

print("Rows (all):", len(fact_cust_product_v2))
print("Rows (active):", len(fact_cust_product_active_v2))
fact_cust_product_v2.head(10)

Rows (all): 34407
Rows (active): 20859


,cust_id,product_group,deposit_category,product_type,balance,active_flag
0,1,Deposit,IBB,CD Long Term,36532.55,Y
1,1,Deposit,NIDDA,Checking,3063.10,Y
2,1,Deposit,IBB,CD Short Term,25385.98,Y
3,1,Deposit,IBB,Money Market,8807.90,Y
4,1,Deposit,IBB,Savings,5870.01,Y
5,1,Credit,N/A,Credit Card,618.11,Y
6,1,Loan,N/A,Home Loan,0.00,N
7,1,Loan,N/A,Auto Loan,39142.99,Y
8,1,Loan,N/A,Personal Loan,0.00,N
9,2,Deposit,IBB,CD Long Term,47539.56,Y


In [7]:
# Export active-only
fact_cust_product_active_v2.to_csv("/data/fact_cust_product.csv", index=False)